# LeetCode 4065 — Rearrange Array by Removing Distinct Values bằng Python

Link đề: https://leetcode.com/problems/rearrange-array-by-removing-distinct-values/ · Độ khó: **Easy** · Chủ đề: Array, Hash Table, Counting, Sorting, Simulation

Skeleton LeetCode:

```python
class Solution:
    def rearrangeArray(self, nums: List[int]) -> List[int]:
        ...
```

Ghi chú: đề và các ví dụ lấy từ đề gốc (Weekly Contest 521, câu 1); mọi đáp án trong notebook đều được kiểm chứng bằng code, kể cả bằng một lời giải độc lập mô phỏng đúng từng vòng của đề.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: lặp lại thao tác "**lấy ra đúng một bản sao của mỗi giá trị khác nhau** còn trong `nums`, nối chúng vào `ans` theo **thứ tự tăng dần**" cho tới khi `nums` hết; trả về `ans`.
2. Biết **mô phỏng trực tiếp** bằng `set`, `sorted`, `list.remove`, và thấy vì sao nó chậm / dễ sai.
3. Nhận ra **bước đột phá**: phần tử `x` xuất hiện `c` lần sẽ có mặt ở các **vòng** `1, 2, ..., c`; tức mỗi bản sao thuộc **vòng bằng thứ tự xuất hiện của nó** (bản sao thứ `k` thuộc vòng `k`). Do đó chỉ cần đếm số lần (`Counter`) rồi **xếp theo (vòng, giá trị)**.
4. Đi từ **mô phỏng `remove`** → **mảng đếm quét lại mọi giá trị** (cách tham khảo) → **chỉ quét các khóa còn sống** → **sắp xếp theo `(vòng, giá trị)`** → **xếp vào các "thùng" theo vòng**.
5. Tránh các lỗi kinh điển: xóa **mọi** bản sao thay vì **một**, sửa list khi đang duyệt, quên sắp tăng trong mỗi vòng (hoặc sắp cả mảng một lần), và dùng thứ tự khóa của `Counter` (thứ tự xuất hiện, không sắp).

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums`. Bạn bắt đầu với mảng rỗng `ans`. Lặp lại thao tác sau cho tới khi `nums` rỗng:

- Xác định **tất cả các giá trị khác nhau** hiện còn trong `nums`.
- **Xóa một bản sao của mỗi giá trị khác nhau** đang có trong `nums`, và **nối** các giá trị đó vào `ans` theo **thứ tự tăng dần**.

Trả về mảng `ans`.

**Ví dụ 1**

- Input: `nums = [3,1,3,2,1,3]`
- Output: `[1,2,3,1,3,3]`
- Giải thích: vòng 1: các giá trị khác nhau là `1, 2, 3` → nối `1, 2, 3`, `nums` còn `[3, 1, 3]`. Vòng 2: các giá trị khác nhau là `1, 3` → nối `1, 3`, `nums` còn `[3]`. Vòng 3: nối `3`, `nums` rỗng. Kết quả `[1,2,3,1,3,3]`.

**Ví dụ 2**

- Input: `nums = [7,7,4,4,4]`
- Output: `[4,7,4,7,4]`
- Giải thích: vòng 1: `4, 7` (còn `[7, 4, 4]`); vòng 2: `4, 7` (còn `[4]`); vòng 3: `4`. Kết quả `[4,7,4,7,4]`.

**Ràng buộc:** `1 <= nums.length <= 100`; `1 <= nums[i] <= 100`.

In [1]:
from typing import List
import random
from collections import Counter

examples = [
    ([3, 1, 3, 2, 1, 3], [1, 2, 3, 1, 3, 3]),
    ([7, 7, 4, 4, 4], [4, 7, 4, 7, 4]),
]
for nums, expected in examples:
    print(f"nums={nums}, kỳ vọng {expected}")

nums=[3, 1, 3, 2, 1, 3], kỳ vọng [1, 2, 3, 1, 3, 3]
nums=[7, 7, 4, 4, 4], kỳ vọng [4, 7, 4, 7, 4]


## 3. Kiến thức nền

### 3.1. `set` — giữ các giá trị khác nhau

`set(nums)` loại các giá trị trùng và chỉ giữ một bản mỗi giá trị; `len(set(nums))` là số giá trị khác nhau. Thứ tự các phần tử trong `set` **không được đảm bảo**, nên phải `sorted(...)` khi cần thứ tự.

In [2]:
nums = [3, 1, 3, 2, 1, 3]
print(set(nums))             # {1, 2, 3}  (thứ tự tùy ý)
print(len(set(nums)))        # 3
print(sorted(set(nums)))     # [1, 2, 3]  (đã sắp tăng dần, là list)

{1, 2, 3}
3
[1, 2, 3]


### 3.2. `sorted` và `list.remove` (chỉ xóa MỘT bản sao đầu tiên)

`sorted(iterable)` trả về **list mới** đã sắp tăng. `lst.remove(x)` xóa **phần tử đầu tiên** bằng `x` trong `lst` (các bản sao sau vẫn còn) và sửa `lst` tại chỗ. Nếu `x` không có, `remove` báo `ValueError`.

In [3]:
lst = [3, 1, 3, 2, 1, 3]
lst.remove(3)
print(lst)                   # [1, 3, 2, 1, 3]  chỉ mất 3 đầu tiên
lst.remove(1)
print(lst)                   # [3, 2, 1, 3]
try:
    lst.remove(9)
except ValueError as e:
    print("ValueError:", e)

[1, 3, 2, 1, 3]
[3, 2, 1, 3]
ValueError: list.remove(x): x not in list


### 3.3. `Counter` và duyệt các cặp `(giá trị, số lần)`

`collections.Counter(nums)` đếm số lần mỗi giá trị xuất hiện. `.items()` cho các cặp `(giá trị, số lần)`. Thứ tự khóa của `Counter` là **thứ tự lần đầu xuất hiện** trong `nums`, **không phải thứ tự tăng dần**: muốn tăng dần phải `sorted(cnt)`.

In [4]:
cnt = Counter([3, 1, 3, 2, 1, 3])
print(cnt)                   # Counter({3: 3, 1: 2, 2: 1})
print(list(cnt))             # [3, 1, 2]  <- thứ tự xuất hiện lần đầu
print(sorted(cnt))           # [1, 2, 3]  <- thứ tự tăng
for x in sorted(cnt):
    print(f"giá trị {x} xuất hiện {cnt[x]} lần")

Counter({3: 3, 1: 2, 2: 1})
[3, 1, 2]
[1, 2, 3]
giá trị 1 xuất hiện 2 lần
giá trị 2 xuất hiện 1 lần
giá trị 3 xuất hiện 3 lần


### 3.4. Không sửa list khi đang duyệt nó

Nếu vừa `for x in lst` vừa `lst.remove(...)`, các phần tử dồn lại và vòng lặp **bỏ qua** phần tử. Đây là lỗi kinh điển. Cách an toàn: duyệt một **bản sao** (`for x in list(lst)`), hoặc duyệt một danh sách khác (ví dụ `sorted(set(lst))`).

In [5]:
lst = [1, 1, 2, 2]
for x in lst:                 # sai: vừa duyệt vừa sửa
    lst.remove(x)
print("sau khi remove khi duyệt:", lst)         # [1, 2] còn sót

lst = [1, 1, 2, 2]
for x in list(lst):           # đúng: duyệt bản sao
    lst.remove(x)
print("sau khi duyệt bản sao:", lst)            # []

sau khi remove khi duyệt: [1, 2]
sau khi duyệt bản sao: []


### 3.5. Sắp xếp danh sách các cặp `(vòng, giá trị)`

`sorted` với list các **tuple** so sánh theo phần tử đầu rồi phần tử thứ hai. Ta sẽ dùng cặp `(vòng, giá trị)`: sắp theo vòng trước, trong cùng vòng thì theo giá trị.

In [6]:
pairs = [(2, 3), (1, 3), (1, 1), (3, 3), (2, 1), (1, 2)]
print(sorted(pairs))         # [(1, 1), (1, 2), (1, 3), (2, 1), (2, 3), (3, 3)]
print([v for _, v in sorted(pairs)])    # [1, 2, 3, 1, 3, 3]

[(1, 1), (1, 2), (1, 3), (2, 1), (2, 3), (3, 3)]
[1, 2, 3, 1, 3, 3]


### 3.6. `list.extend`, `append` và list của list ("thùng")

`ans.append(x)` thêm một phần tử; `ans.extend(iterable)` thêm cả một dãy. Một **list của list** `buckets = [[], [], []]` dùng để xếp phần tử vào từng "thùng" theo chỉ số vòng.

In [7]:
ans = []
ans.append(5)
ans.extend([1, 2, 3])
print(ans)                    # [5, 1, 2, 3]

buckets = [[] for _ in range(3)]       # 3 thùng rỗng (không dùng [[]] * 3!)
buckets[0].append(1)
buckets[2].append(3)
print(buckets)                # [[1], [], [3]]

[5, 1, 2, 3]
[[1], [], [3]]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Làm tay ví dụ 1: `nums = [3, 1, 3, 2, 1, 3]`. Số lần: `1` xuất hiện 2 lần, `2` xuất hiện 1 lần, `3` xuất hiện 3 lần.

| vòng | các giá trị khác nhau còn lại | nối vào `ans` | `nums` còn lại |
|---|---|---|---|
| 1 | `1, 2, 3` | `1, 2, 3` | `[3, 1, 3]` |
| 2 | `1, 3` | `1, 3` | `[3]` |
| 3 | `3` | `3` | `[]` |

Quan sát quan trọng: **thứ tự các phần tử trong `nums` không ảnh hưởng gì** tới kết quả — chỉ có **số lần xuất hiện** mỗi giá trị quan trọng. Giá trị `3` (3 lần) có mặt ở cả 3 vòng, `1` (2 lần) ở vòng 1 và 2, `2` (1 lần) chỉ ở vòng 1.

Vậy `ans` = vòng 1 (mọi giá trị có số lần ≥ 1) + vòng 2 (mọi giá trị có số lần ≥ 2) + vòng 3 (số lần ≥ 3) + ..., mỗi vòng sắp tăng.

In [8]:
nums = [3, 1, 3, 2, 1, 3]
cnt = Counter(nums)
max_round = max(cnt.values())
for k in range(1, max_round + 1):
    block = sorted(x for x in cnt if cnt[x] >= k)
    print(f"vòng {k}: các giá trị có số lần >= {k}: {block}")

vòng 1: các giá trị có số lần >= 1: [1, 2, 3]
vòng 2: các giá trị có số lần >= 2: [1, 3]
vòng 3: các giá trị có số lần >= 3: [3]


## 5. Cách 1 — Brute Force / baseline: mô phỏng đúng như đề

**Ý tưởng.** Làm đúng mô tả của đề, từng bước:

1. Sao chép `nums` (để không sửa đầu vào).
2. Trong khi list còn phần tử: lấy `distinct = sorted(set(nums))`; nối vào `ans`; rồi với mỗi `x` trong `distinct` gọi `nums.remove(x)` để bỏ **một** bản sao.

In [9]:
def rearrange_simulate(nums: list[int]) -> list[int]:
    nums = list(nums)                        # sao chép, không phá đầu vào
    ans = []
    while nums:                               # còn phần tử thì còn vòng
        distinct = sorted(set(nums))          # các giá trị khác nhau, tăng dần
        ans.extend(distinct)
        for x in distinct:                    # duyệt list 'distinct', không phải 'nums'
            nums.remove(x)                    # chỉ xóa MỘT bản sao
    return ans


for nums, expected in examples:
    print(f"nums={nums} -> {rearrange_simulate(nums)} (kỳ vọng {expected})")

nums=[3, 1, 3, 2, 1, 3] -> [1, 2, 3, 1, 3, 3] (kỳ vọng [1, 2, 3, 1, 3, 3])
nums=[7, 7, 4, 4, 4] -> [4, 7, 4, 7, 4] (kỳ vọng [4, 7, 4, 7, 4])


**Độ phức tạp (Cách 1).** Gọi `n = len(nums)`, `d` là số giá trị khác nhau, `R` là số vòng (`R` bằng tần suất lớn nhất). Mỗi vòng tạo `set` và sắp xếp: `O(n + d log d)`; rồi `d` lần `remove` mỗi lần `O(n)`. Tổng số lần `remove` qua mọi vòng đúng bằng `n` (mỗi phần tử bị xóa một lần), mỗi lần `O(n)`; tạo `set` ở mỗi vòng tốn thêm `O(n · R) ≤ O(n²)`. Vậy **Time `O(n²)`** (cộng `O(n log n)` cho các lần sắp); với `n ≤ 100` rất nhanh. **Space `O(n)`**.

**Nút thắt.** Ta tìm lại các giá trị khác nhau và `remove` (tốn `O(n)`) ở mỗi vòng. Thực ra **thông tin đủ để xếp `ans`** chỉ là số lần xuất hiện của từng giá trị: không cần xóa gì cả.

## 6. Ý tưởng thuật toán / bước đột phá

**Nhìn theo từng giá trị.** Giá trị `x` xuất hiện `c` lần thì nó bị lấy đúng **một lần mỗi vòng** trong các vòng `1, 2, ..., c` (vì nó còn trong `nums` đúng `c` vòng). Vậy bản sao thứ `k` (đếm từ 1) của `x` thuộc **vòng `k`**.

Do đó `ans` là các cặp `(vòng, giá trị)` được liệt kê theo thứ tự: **vòng nhỏ trước**, trong cùng vòng thì **giá trị nhỏ trước**:

```
(1, x) với mọi x có cnt[x] >= 1   (theo x tăng)
(2, x) với mọi x có cnt[x] >= 2
...
```

Điều này cho phép:

- **Đếm rồi quét vòng:** với mỗi vòng, quét các giá trị theo thứ tự tăng và lấy giá trị còn đếm > 0 (Cách 2, Cách 3);
- **Sắp xếp các cặp `(vòng, giá trị)`** một lượt (Cách 4);
- **Xếp vào thùng theo vòng** (Cách 5).

In [10]:
cnt = Counter([3, 1, 3, 2, 1, 3])
pairs = [(k, x) for x, c in cnt.items() for k in range(1, c + 1)]
print("các cặp (vòng, giá trị):", pairs)
print("sau khi sắp:", sorted(pairs))
print("ans =", [x for _, x in sorted(pairs)])

các cặp (vòng, giá trị): [(1, 3), (2, 3), (3, 3), (1, 1), (2, 1), (1, 2)]
sau khi sắp: [(1, 1), (1, 2), (1, 3), (2, 1), (2, 3), (3, 3)]
ans = [1, 2, 3, 1, 3, 3]


## 7. Các cách tiếp theo

### Cách 2 — Mảng đếm và quét lại mọi giá trị trong mỗi vòng (cách của lời giải tham khảo)

**Giải thích.** Do `1 <= nums[i] <= 100`, tạo `cnt` là mảng độ dài `max(nums) + 1`. Lặp: trong khi `ans` chưa đủ `len(nums)` phần tử, quét `x` từ `1` đến `max`: nếu `cnt[x] > 0` thì thêm `x` vào `ans` và giảm `cnt[x]`. Mỗi lần quét xong là một vòng, các giá trị thêm vào đã tăng dần nhờ thứ tự quét.

In [11]:
def rearrange_count_array(nums: list[int]) -> list[int]:
    top = max(nums)
    cnt = [0] * (top + 1)
    for x in nums:
        cnt[x] += 1                                  # cnt[x] = số lần x xuất hiện
    ans = []
    while len(ans) < len(nums):                      # mỗi lượt while là một vòng
        for x in range(1, top + 1):                  # quét từ nhỏ đến lớn
            if cnt[x] > 0:
                ans.append(x)
                cnt[x] -= 1
    return ans


for nums, expected in examples:
    print(f"nums={nums} -> {rearrange_count_array(nums)} (kỳ vọng {expected})")

nums=[3, 1, 3, 2, 1, 3] -> [1, 2, 3, 1, 3, 3] (kỳ vọng [1, 2, 3, 1, 3, 3])
nums=[7, 7, 4, 4, 4] -> [4, 7, 4, 7, 4] (kỳ vọng [4, 7, 4, 7, 4])


In [12]:
# Code nộp LeetCode cho Cách 2
from typing import List


class Solution:
    def rearrangeArray(self, nums: List[int]) -> List[int]:
        top = max(nums)
        cnt = [0] * (top + 1)
        for x in nums:
            cnt[x] += 1
        ans = []
        while len(ans) < len(nums):
            for x in range(1, top + 1):
                if cnt[x]:
                    ans.append(x)
                    cnt[x] -= 1
        return ans

**Độ phức tạp Cách 2:** `R` vòng, mỗi vòng quét `m = max(nums)` giá trị: **Time `O(R · m)`** (tối đa `n · m`), **Space `O(m)`**. Với `n, m ≤ 100` là tối đa `10⁴` thao tác.

### Cách 3 — Chỉ quét các khóa còn "sống"

**Giải thích.** Thay vì quét `1..max` mỗi vòng (nhiều giá trị đã hết), dùng `Counter` và `keys = sorted(cnt)` (các giá trị khác nhau, tăng dần). Mỗi vòng: thêm mọi `x` trong `keys` vào `ans`, giảm `cnt[x]`, rồi lọc `keys` chỉ giữ những `x` còn `cnt[x] > 0`. Tổng công việc qua mọi vòng đúng bằng `n` (mỗi phần tử thêm một lần).

In [13]:
def rearrange_live_keys(nums: list[int]) -> list[int]:
    cnt = Counter(nums)
    keys = sorted(cnt)                               # các giá trị khác nhau, tăng dần
    ans = []
    while keys:
        for x in keys:
            ans.append(x)
            cnt[x] -= 1
        keys = [x for x in keys if cnt[x] > 0]       # bỏ các giá trị đã hết (giữ nguyên thứ tự tăng)
    return ans


for nums, expected in examples:
    print(f"nums={nums} -> {rearrange_live_keys(nums)} (kỳ vọng {expected})")

nums=[3, 1, 3, 2, 1, 3] -> [1, 2, 3, 1, 3, 3] (kỳ vọng [1, 2, 3, 1, 3, 3])
nums=[7, 7, 4, 4, 4] -> [4, 7, 4, 7, 4] (kỳ vọng [4, 7, 4, 7, 4])


**Độ phức tạp Cách 3:** **Time `O(n + d log d)`** (sắp `d` khóa một lần; tổng số lần thêm vào `ans` là `n`, và lọc khóa cũng tổng `O(n)`), **Space `O(n)`**.

### Cách 4 — Sắp xếp các cặp `(vòng, giá trị)`

**Giải thích.** Với mỗi giá trị `x` có `c` lần, sinh các cặp `(1, x), (2, x), ..., (c, x)`; gom tất cả và `sorted`; lấy phần giá trị. Cách này **không mô phỏng**, chỉ dựa vào quan sát "bản sao thứ `k` thuộc vòng `k`".

In [14]:
def rearrange_sort_pairs(nums: list[int]) -> list[int]:
    cnt = Counter(nums)
    pairs = [(k, x) for x, c in cnt.items() for k in range(1, c + 1)]
    pairs.sort()                                     # theo vòng, rồi theo giá trị
    return [x for _, x in pairs]


for nums, expected in examples:
    print(f"nums={nums} -> {rearrange_sort_pairs(nums)} (kỳ vọng {expected})")

nums=[3, 1, 3, 2, 1, 3] -> [1, 2, 3, 1, 3, 3] (kỳ vọng [1, 2, 3, 1, 3, 3])
nums=[7, 7, 4, 4, 4] -> [4, 7, 4, 7, 4] (kỳ vọng [4, 7, 4, 7, 4])


**Độ phức tạp Cách 4:** có tổng `n` cặp: **Time `O(n log n)`**, **Space `O(n)`**.

### Cách 5 — Xếp vào "thùng" theo vòng rồi nối lại

**Giải thích.** Tạo `R = max(cnt.values())` thùng (vòng `1..R`). Duyệt các giá trị theo thứ tự **tăng**; với mỗi `x` có `c` lần, thêm `x` vào thùng `1..c`. Vì ta duyệt `x` tăng dần, mỗi thùng tự động tăng dần. Nối các thùng theo thứ tự vòng.

In [15]:
def rearrange_buckets(nums: list[int]) -> list[int]:
    cnt = Counter(nums)
    rounds = max(cnt.values())                       # số vòng = tần suất lớn nhất
    buckets = [[] for _ in range(rounds)]            # buckets[k - 1] là vòng k
    for x in sorted(cnt):                            # giá trị tăng dần
        for k in range(cnt[x]):
            buckets[k].append(x)                     # bản sao thứ k + 1 của x thuộc vòng k + 1
    ans = []
    for block in buckets:
        ans.extend(block)
    return ans


for nums, expected in examples:
    print(f"nums={nums} -> {rearrange_buckets(nums)} (kỳ vọng {expected})")

nums=[3, 1, 3, 2, 1, 3] -> [1, 2, 3, 1, 3, 3] (kỳ vọng [1, 2, 3, 1, 3, 3])
nums=[7, 7, 4, 4, 4] -> [4, 7, 4, 7, 4] (kỳ vọng [4, 7, 4, 7, 4])


**Độ phức tạp Cách 5:** **Time `O(n + d log d)`**, **Space `O(n)`**.

> Cách 2 trùng với lời giải tham khảo, đủ nhanh với ràng buộc. Cách 3 và Cách 5 tối ưu hơn về lý thuyết. Cách 1 và Cách 4 giúp thấy hai góc nhìn: mô phỏng và "vòng = thứ tự xuất hiện".

## 8. TRACE từng bước

Ta theo dõi Cách 3 (khóa còn sống) trên `nums = [3, 1, 3, 2, 1, 3]` và Cách 5 (thùng) trên `nums = [7, 7, 4, 4, 4]`.

In [16]:
def trace_live_keys(nums):
    cnt = Counter(nums)
    keys = sorted(cnt)
    ans = []
    print(f"nums={nums}")
    print(f"cnt={dict(cnt)}, keys ban đầu (đã sắp)={keys}")
    round_no = 0
    while keys:
        round_no += 1
        taken = []
        for x in keys:
            ans.append(x)
            cnt[x] -= 1
            taken.append(x)
        keys = [x for x in keys if cnt[x] > 0]
        print(f"  vòng {round_no}: lấy {taken}; cnt={dict(cnt)}; khóa còn sống={keys}; ans={ans}")
    print("  => ans =", ans)
    return ans


assert trace_live_keys([3, 1, 3, 2, 1, 3]) == [1, 2, 3, 1, 3, 3]

nums=[3, 1, 3, 2, 1, 3]
cnt={3: 3, 1: 2, 2: 1}, keys ban đầu (đã sắp)=[1, 2, 3]
  vòng 1: lấy [1, 2, 3]; cnt={3: 2, 1: 1, 2: 0}; khóa còn sống=[1, 3]; ans=[1, 2, 3]
  vòng 2: lấy [1, 3]; cnt={3: 1, 1: 0, 2: 0}; khóa còn sống=[3]; ans=[1, 2, 3, 1, 3]
  vòng 3: lấy [3]; cnt={3: 0, 1: 0, 2: 0}; khóa còn sống=[]; ans=[1, 2, 3, 1, 3, 3]
  => ans = [1, 2, 3, 1, 3, 3]


In [17]:
def trace_buckets(nums):
    cnt = Counter(nums)
    rounds = max(cnt.values())
    buckets = [[] for _ in range(rounds)]
    print(f"nums={nums}, cnt={dict(cnt)}, số vòng = {rounds}")
    for x in sorted(cnt):
        for k in range(cnt[x]):
            buckets[k].append(x)
        print(f"  thêm giá trị {x} ({cnt[x]} lần) vào thùng 1..{cnt[x]}: {buckets}")
    ans = [x for block in buckets for x in block]
    print("  => ans =", ans)
    return ans


assert trace_buckets([7, 7, 4, 4, 4]) == [4, 7, 4, 7, 4]

nums=[7, 7, 4, 4, 4], cnt={7: 2, 4: 3}, số vòng = 3
  thêm giá trị 4 (3 lần) vào thùng 1..3: [[4], [4], [4]]
  thêm giá trị 7 (2 lần) vào thùng 1..2: [[4, 7], [4, 7], [4]]
  => ans = [4, 7, 4, 7, 4]


Bảng tóm tắt (Cách 3, ví dụ 1):

| vòng | `keys` đầu vòng | lấy | `cnt` sau | `keys` còn sống | `ans` |
|---|---|---|---|---|---|
| 1 | `[1, 2, 3]` | `1, 2, 3` | `{1:1, 2:0, 3:2}` | `[1, 3]` | `[1, 2, 3]` |
| 2 | `[1, 3]` | `1, 3` | `{1:0, 2:0, 3:1}` | `[3]` | `[1, 2, 3, 1, 3]` |
| 3 | `[3]` | `3` | `{1:0, 2:0, 3:0}` | `[]` | `[1, 2, 3, 1, 3, 3]` |

Giá trị `2` bị loại khỏi `keys` ngay sau vòng 1 (đếm về 0) nên các vòng sau không phải đụng tới nó.

## 9. Code nộp LeetCode

Bản nộp dùng Cách 3 (chỉ quét khóa còn sống, dùng `Counter` và `sorted`). Đúng class `Solution`, đúng method `rearrangeArray(self, nums: List[int]) -> List[int]`.

In [18]:
from collections import Counter
from typing import List


class Solution:
    def rearrangeArray(self, nums: List[int]) -> List[int]:
        cnt = Counter(nums)                      # giá trị -> số lần còn lại
        keys = sorted(cnt)                       # các giá trị khác nhau, tăng dần
        ans = []
        while keys:                              # mỗi lượt là một vòng của đề
            for x in keys:
                ans.append(x)                    # lấy một bản sao của mỗi giá trị khác nhau
                cnt[x] -= 1
            keys = [x for x in keys if cnt[x] > 0]    # giữ lại những giá trị còn bản sao
        return ans


for nums, expected in examples:
    assert Solution().rearrangeArray(nums) == expected
print("PASS ví dụ đề cho class Solution")

PASS ví dụ đề cho class Solution


## 10. Vì sao thuật toán đúng

- **Mỗi giá trị bị lấy một lần mỗi vòng:** ở mỗi vòng, đề lấy đúng một bản sao của **mỗi giá trị đang còn trong `nums`**. Giá trị `x` có `cnt[x]` bản sao, nên nó còn trong `nums` đúng `cnt[x]` vòng đầu tiên, bị lấy đúng một lần mỗi vòng.
- **Vòng `k` gồm chính xác các giá trị có `cnt[x] ≥ k`:** vì sau `k − 1` vòng, `x` còn `cnt[x] − (k − 1)` bản sao, dương khi `cnt[x] ≥ k`.
- **Thứ tự trong vòng:** đề yêu cầu tăng dần; `keys` luôn tăng dần vì ta sắp một lần rồi chỉ **lọc** (lọc giữ nguyên thứ tự).
- **Không cần `nums` thật:** hành động "xóa một bản sao" chỉ đổi số lần của giá trị đó (`cnt[x] -= 1`); thứ tự các phần tử trong `nums` không bao giờ ảnh hưởng tới kết quả.
- **Dừng:** khi `keys` rỗng, mọi `cnt[x] = 0`, tức `nums` rỗng; tổng số phần tử thêm vào `ans` bằng `n`.
- **Cách 2 tương đương:** vòng lặp `while len(ans) < len(nums)` chạy đúng `R` lần, mỗi lần quét đủ mọi giá trị tăng dần và lấy giá trị còn `cnt > 0`.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(nums)`, `d` là số giá trị khác nhau, `R` là tần suất lớn nhất (= số vòng), `m = max(nums)`.

- **Cách 1 (mô phỏng `remove`):** Time `O(n²)` (tổng `n` lần `remove`, mỗi lần `O(n)`, cộng việc tạo `set` mỗi vòng), Space `O(n)`.
- **Cách 2 (mảng đếm quét `1..m`):** Time `O(R · m)` ≤ `O(n · m)`, Space `O(m)`.
- **Cách 3 (khóa còn sống):** Time `O(n + d log d)`, Space `O(n)`.
- **Cách 4 (sắp các cặp):** Time `O(n log n)`, Space `O(n)`.
- **Cách 5 (thùng theo vòng):** Time `O(n + d log d)`, Space `O(n)`.
- Với mọi cách, trường hợp trung bình và xấu nhất như nhau (không phụ thuộc cấu trúc băm hay ngẫu nhiên). Không thể tốt hơn `O(n)` vì phải sinh `n` phần tử.

## 12. Bộ test

Ta gom các cách vào từ điển `impls` và viết một **lời giải độc lập** `oracle` mô phỏng đúng từng vòng bằng cách **dựng lại list mới** (dùng tập `used` đánh dấu giá trị đã lấy trong vòng, không dùng `remove`).

In [19]:
sol = Solution()
impls = {
    "simulate": rearrange_simulate,
    "count_array": rearrange_count_array,
    "live_keys": rearrange_live_keys,
    "sort_pairs": rearrange_sort_pairs,
    "buckets": rearrange_buckets,
    "submit": sol.rearrangeArray,
}


def oracle(nums):
    """Mô phỏng theo đề, dựng lại list mỗi vòng (không dùng remove/Counter)."""
    current = list(nums)
    ans = []
    while current:
        used = set()
        rest = []
        taken = []
        for x in current:
            if x not in used:
                used.add(x)
                taken.append(x)                 # lấy bản sao đầu tiên của x trong vòng này
            else:
                rest.append(x)                  # các bản sao sau của x ở lại
        taken.sort()
        ans.extend(taken)
        current = rest
    return ans


for nums, expected in examples:
    assert oracle(nums) == expected
    for name, fn in impls.items():
        assert fn(list(nums)) == expected, (name, nums, fn(list(nums)), expected)
print("PASS ví dụ của đề (mọi cách và oracle độc lập)")

PASS ví dụ của đề (mọi cách và oracle độc lập)


### 12.1. Các trường hợp biên

In [20]:
def check_all(nums, expected, label):
    assert oracle(nums) == expected, (label, "oracle", oracle(nums), expected)
    for name, fn in impls.items():
        original = list(nums)
        got = fn(original)
        assert got == expected, (label, name, got, expected)
        assert original == nums, (label, name, "đã sửa đầu vào")
    print("PASS", label, "->", expected)


check_all([5], [5], "n = 1")
check_all([3, 1, 2], [1, 2, 3], "mọi giá trị khác nhau: một vòng, chỉ là sắp tăng")
check_all([2, 2, 2], [2, 2, 2], "toàn một giá trị")
check_all([3, 3, 1, 1], [1, 3, 1, 3], "hai giá trị mỗi giá trị hai lần")
check_all([100, 1, 100], [1, 100, 100], "biên giá trị 1 và 100")
check_all([5, 4, 3, 2, 1], [1, 2, 3, 4, 5], "giảm dần, mọi giá trị khác nhau")
check_all([2, 5, 2, 9, 5, 2], [2, 5, 9, 2, 5, 2], "ba giá trị với tần suất 3, 2, 1")
check_all([4, 4, 4, 1], [1, 4, 4, 4], "một giá trị nhiều lần, một giá trị một lần")
check_all([1] * 100, [1] * 100, "n = 100, toàn một giá trị")
check_all(list(range(1, 101)), list(range(1, 101)), "n = 100, mọi giá trị khác nhau")

PASS n = 1 -> [5]
PASS mọi giá trị khác nhau: một vòng, chỉ là sắp tăng -> [1, 2, 3]
PASS toàn một giá trị -> [2, 2, 2]
PASS hai giá trị mỗi giá trị hai lần -> [1, 3, 1, 3]
PASS biên giá trị 1 và 100 -> [1, 100, 100]
PASS giảm dần, mọi giá trị khác nhau -> [1, 2, 3, 4, 5]
PASS ba giá trị với tần suất 3, 2, 1 -> [2, 5, 9, 2, 5, 2]
PASS một giá trị nhiều lần, một giá trị một lần -> [1, 4, 4, 4]
PASS n = 100, toàn một giá trị -> [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
PASS n = 100, mọi giá trị khác nhau -> [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60

### 12.2. Tính chất và phản ví dụ

In [21]:
random.seed(4065)
for _ in range(500):
    n = random.randint(1, 30)
    nums = [random.randint(1, 6) for _ in range(n)]
    ans = sol.rearrangeArray(nums)
    # Tính chất 1: ans là hoán vị của nums
    assert sorted(ans) == sorted(nums)
    # Tính chất 2: ans chia được thành các đoạn tăng chặt tối đại; số đoạn = tần suất lớn nhất
    runs = 1 + sum(1 for i in range(len(ans) - 1) if ans[i] >= ans[i + 1])
    assert runs == max(Counter(nums).values()), (nums, ans)
    # Tính chất 3: đoạn đầu tiên là tập các giá trị khác nhau, đã sắp
    first_len = len(set(nums))
    assert ans[:first_len] == sorted(set(nums))
    # Tính chất 4: thứ tự nums không ảnh hưởng đến kết quả
    shuffled = nums[:]
    random.shuffle(shuffled)
    assert sol.rearrangeArray(shuffled) == ans
print("PASS tính chất: hoán vị; số đoạn tăng = tần suất lớn nhất; đoạn đầu = giá trị khác nhau đã sắp; không phụ thuộc thứ tự nums")

# Phản ví dụ: sắp tăng cả mảng không đúng
bad = sorted([3, 1, 3, 2, 1, 3])
print("sorted(nums) =", bad, "| đúng:", sol.rearrangeArray([3, 1, 3, 2, 1, 3]))
assert bad != sol.rearrangeArray([3, 1, 3, 2, 1, 3])

PASS tính chất: hoán vị; số đoạn tăng = tần suất lớn nhất; đoạn đầu = giá trị khác nhau đã sắp; không phụ thuộc thứ tự nums
sorted(nums) = [1, 1, 2, 3, 3, 3] | đúng: [1, 2, 3, 1, 3, 3]


### 12.3. Đối chiếu ngẫu nhiên (có seed) với oracle độc lập

In [22]:
random.seed(2026)
for trial in range(3000):
    n = random.randint(1, 20)
    hi = random.choice([2, 3, 5, 10, 100])
    nums = [random.randint(1, hi) for _ in range(n)]
    expected = oracle(nums)
    for name, fn in impls.items():
        assert fn(list(nums)) == expected, (name, nums, fn(list(nums)), expected)

for trial in range(200):                           # n tới 100
    n = random.randint(60, 100)
    hi = random.choice([3, 10, 50, 100])
    nums = [random.randint(1, hi) for _ in range(n)]
    expected = oracle(nums)
    for name, fn in impls.items():
        assert fn(list(nums)) == expected, (name, hi)
print("PASS 3200 input ngẫu nhiên: mọi cách khớp oracle độc lập")

PASS 3200 input ngẫu nhiên: mọi cách khớp oracle độc lập


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Xóa mọi bản sao của giá trị thay vì chỉ một

In [23]:
def sai_xoa_het(nums):
    nums = list(nums)
    ans = []
    while nums:
        distinct = sorted(set(nums))
        ans.extend(distinct)
        nums = [x for x in nums if x not in distinct]      # xóa HẾT mọi bản sao
    return ans


print("sai:", sai_xoa_het([3, 1, 3, 2, 1, 3]), "| đúng:", Solution().rearrangeArray([3, 1, 3, 2, 1, 3]))

sai: [1, 2, 3] | đúng: [1, 2, 3, 1, 3, 3]


**Vì sao sai:** đề chỉ cho xóa **một** bản sao của mỗi giá trị mỗi vòng. Nếu xóa tất cả thì sau vòng 1 `nums` rỗng và `ans = [1, 2, 3]` thiếu các bản sao còn lại. Dùng `nums.remove(x)` (xóa một) hoặc giảm đếm `cnt[x] -= 1`.

### Lỗi 2: Sửa list khi đang duyệt nó

In [24]:
def sai_sua_khi_duyet(nums):
    nums = list(nums)
    ans = []
    while nums:
        distinct = sorted(set(nums))
        ans.extend(distinct)
        pending = set(distinct)
        for x in nums:                        # sai: vừa duyệt nums vừa sửa nums
            if x in pending:
                nums.remove(x)
                pending.discard(x)
    return ans


print("sai:", sai_sua_khi_duyet([3, 1, 3, 2, 1, 3]), "| đúng:", Solution().rearrangeArray([3, 1, 3, 2, 1, 3]))

sai: [1, 2, 3, 1, 3, 1, 3, 3] | đúng: [1, 2, 3, 1, 3, 3]


**Vì sao sai:** khi `nums.remove(x)` chạy giữa lúc `for x in nums`, các phần tử phía sau dồn về phía trước và vòng lặp **bỏ qua** phần tử kế tiếp. Với ví dụ 1 một số bản sao không bị xóa đúng lúc, nên `ans` thừa phần tử (độ dài 8 thay vì 6). Hãy duyệt `sorted(set(nums))` (một list khác) hoặc dùng bộ đếm, như Cách 1 và Cách 3.

### Lỗi 3: Sắp xếp cả mảng một lần

In [25]:
def sai_sap_ca_mang(nums):
    return sorted(nums)


print("sai:", sai_sap_ca_mang([3, 1, 3, 2, 1, 3]), "| đúng:", Solution().rearrangeArray([3, 1, 3, 2, 1, 3]))

sai: [1, 1, 2, 3, 3, 3] | đúng: [1, 2, 3, 1, 3, 3]


**Vì sao sai:** kết quả đúng chia thành **các vòng**, mỗi vòng tăng dần nhưng giữa các vòng quay về giá trị nhỏ (`1, 2, 3, 1, 3, 3`). Sắp cả mảng cho `1, 1, 2, 3, 3, 3` chỉ đúng khi mọi giá trị trùng nhau hoặc khi chỉ có một vòng.

### Lỗi 4: Dùng thứ tự khóa của `Counter` (không tăng dần)

In [26]:
def sai_thu_tu_counter(nums):
    cnt = Counter(nums)
    keys = list(cnt)                          # thứ tự xuất hiện lần đầu, KHÔNG sắp tăng
    ans = []
    while keys:
        for x in keys:
            ans.append(x)
            cnt[x] -= 1
        keys = [x for x in keys if cnt[x] > 0]
    return ans


print("sai:", sai_thu_tu_counter([3, 1, 3, 2, 1, 3]), "| đúng:", Solution().rearrangeArray([3, 1, 3, 2, 1, 3]))

sai: [3, 1, 2, 3, 1, 3] | đúng: [1, 2, 3, 1, 3, 3]


**Vì sao sai:** `Counter` (như `dict`) giữ khóa theo **thứ tự thêm vào**, tức thứ tự xuất hiện lần đầu: `3, 1, 2`. Đề yêu cầu **tăng dần** trong mỗi vòng, nên phải `sorted(cnt)`. Lỗi này khó thấy vì với một số input (đã tăng sẵn) vẫn cho kết quả đúng.

## 14. Mẹo làm nhanh & nhận dạng pattern

- Thấy "mỗi lượt lấy một bản sao của mỗi giá trị khác nhau": nghĩ ngay **"bản sao thứ `k` thuộc vòng `k`"** và **đếm tần suất** (`Counter`) thay vì mô phỏng xóa.
- Nếu cần giữ thứ tự tăng dần: **sắp các khóa một lần** rồi chỉ **lọc** (giữ nguyên thứ tự) — rẻ hơn sắp lại mỗi vòng.
- Giá trị nhỏ (≤ 100) cho phép dùng **mảng đếm** (đơn giản nhất, đủ nhanh).
- Không sửa list đang duyệt; không dùng `[[]] * k` để tạo thùng (dùng `[[] for _ in range(k)]`).
- Thứ tự phần tử trong input không ảnh hưởng: chỉ có **bộ đếm** quan trọng; dùng để rút gọn bài và để viết test hoán vị.
- Với các bài kiểu "chia mảng thành các nhóm giá trị khác nhau" (LeetCode 2610), cùng ý tưởng: `max tần suất` = số nhóm.

## 15. Biến thể của bài

1. **Thứ tự giảm dần** trong mỗi vòng.
2. **Trả về các vòng** dưới dạng list của list: `[[1,2,3],[1,3],[3]]` (LeetCode 2610).
3. **Chỉ đếm số vòng** (tần suất lớn nhất).
4. **Giữ thứ tự xuất hiện đầu tiên** thay vì sắp tăng (mỗi vòng lấy theo thứ tự xuất hiện lần đầu trong `nums`).
5. **Lấy `k` bản sao** của mỗi giá trị mỗi vòng thay vì 1.

In [27]:
def rearrange_desc(nums):
    """Biến thể 1: mỗi vòng giảm dần."""
    cnt = Counter(nums)
    keys = sorted(cnt, reverse=True)
    ans = []
    while keys:
        for x in keys:
            ans.append(x)
            cnt[x] -= 1
        keys = [x for x in keys if cnt[x] > 0]
    return ans


def rearrange_rounds(nums):
    """Biến thể 2: trả về từng vòng."""
    cnt = Counter(nums)
    rounds = [[] for _ in range(max(cnt.values()))]
    for x in sorted(cnt):
        for k in range(cnt[x]):
            rounds[k].append(x)
    return rounds


def rearrange_num_rounds(nums):
    """Biến thể 3."""
    return max(Counter(nums).values())


def rearrange_first_seen(nums):
    """Biến thể 4: mỗi vòng theo thứ tự xuất hiện lần đầu."""
    cnt = Counter(nums)
    keys = list(cnt)               # thứ tự xuất hiện lần đầu
    ans = []
    while keys:
        for x in keys:
            ans.append(x)
            cnt[x] -= 1
        keys = [x for x in keys if cnt[x] > 0]
    return ans


def rearrange_k_copies(nums, k):
    """Biến thể 5: mỗi vòng lấy tối đa k bản sao của mỗi giá trị."""
    cnt = Counter(nums)
    keys = sorted(cnt)
    ans = []
    while keys:
        for x in keys:
            take = min(k, cnt[x])
            ans.extend([x] * take)
            cnt[x] -= take
        keys = [x for x in keys if cnt[x] > 0]
    return ans


assert rearrange_desc([3, 1, 3, 2, 1, 3]) == [3, 2, 1, 3, 1, 3]
assert rearrange_rounds([3, 1, 3, 2, 1, 3]) == [[1, 2, 3], [1, 3], [3]]
assert rearrange_num_rounds([7, 7, 4, 4, 4]) == 3
assert rearrange_first_seen([3, 1, 3, 2, 1, 3]) == [3, 1, 2, 3, 1, 3]
assert rearrange_k_copies([3, 1, 3, 2, 1, 3], 1) == Solution().rearrangeArray([3, 1, 3, 2, 1, 3])
assert rearrange_k_copies([3, 1, 3, 2, 1, 3], 2) == [1, 1, 2, 3, 3, 3]
print("PASS biến thể")

PASS biến thể


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mô phỏng `remove` | mỗi vòng `sorted(set)` rồi `remove` | `O(n²)` | `O(n)` | hiểu đề; oracle đối chiếu |
| 2. Mảng đếm quét `1..m` | `cnt[x]`, quét lại mọi giá trị mỗi vòng | `O(R·m)` | `O(m)` | **cách tham khảo**, rất đơn giản |
| 3. Khóa còn sống | `sorted(cnt)` một lần, mỗi vòng lọc | `O(n + d log d)` | `O(n)` | **cách nộp bài** |
| 4. Sắp cặp `(vòng, giá trị)` | bản sao thứ `k` thuộc vòng `k` | `O(n log n)` | `O(n)` | thấy được cấu trúc "vòng" |
| 5. Thùng theo vòng | `buckets[k].append(x)` | `O(n + d log d)` | `O(n)` | cần trả về từng vòng riêng |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `sorted(set([3, 1, 3]))`, `Counter("aab")["a"]`, `[5, 5, 6].remove(5)` rồi in list, `Solution().rearrangeArray([2, 2, 2])`, `Solution().rearrangeArray([3, 1, 2])`.

**Bài 2 (tính tay).** Tính `ans` cho `nums = [5, 2, 5, 2, 2, 9]` bằng cách liệt kê từng vòng. Làm lại với `nums = [4, 4, 4, 1]`.

**Bài 3 (điền TODO).** Hoàn thành `rearrange_todo(nums)` theo Cách 3.

In [28]:
def rearrange_todo(nums):
    # TODO 1: cnt = Counter(nums) và keys = sorted(cnt)
    # TODO 2: trong khi keys còn phần tử, với mỗi x trong keys: thêm x vào ans và giảm cnt[x]
    # TODO 3: sau mỗi vòng, giữ lại trong keys chỉ những x có cnt[x] > 0
    # TODO 4: trả về ans
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (các TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `rearrange_pairs2(nums)` theo Cách 4 (sắp các cặp `(vòng, giá trị)`) mà không nhìn code mẫu. Kiểm tra bằng hai ví dụ của đề.

**Bài 5 (biến thể / mini challenge).** (a) Viết `rearrange_desc2(nums)`: mỗi vòng sắp **giảm dần**. (b) Viết `rounds_of(nums)` trả về list các vòng, ví dụ `[[1, 2, 3], [1, 3], [3]]`; và `count_rounds(nums)` trả về số vòng.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `set` bỏ trùng; `Counter` đếm số lần; `remove` chỉ xóa một bản sao đầu tiên. Bài 2: viết ra các vòng, mỗi vòng lấy mỗi giá trị khác nhau một lần.
- **Mức 2 (vừa):** Bài 1: `[1, 3]`, `2`, `[5, 6]`, `[2, 2, 2]`, `[1, 2, 3]`. Bài 2: `[5,2,5,2,2,9]`: vòng 1 `2,5,9`; còn `[5,2]`: vòng 2 `2,5`; còn `[2]`: vòng 3 `2`; kết quả `[2,5,9,2,5,2]`. `[4,4,4,1]`: vòng 1 `1,4`; vòng 2 `4`; vòng 3 `4`; kết quả `[1,4,4,4]`. Bài 3: bốn TODO trong comment.
- **Mức 3 (sâu):** Bài 4: `pairs = [(k, x) for x, c in cnt.items() for k in range(1, c + 1)]`, `pairs.sort()`. Bài 5(a): `sorted(cnt, reverse=True)`. Bài 5(b): thùng thứ `k` (từ 0) chứa các `x` có `cnt[x] > k`; số vòng = `max(cnt.values())`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [29]:
# Bài 1
assert sorted(set([3, 1, 3])) == [1, 3]
assert Counter("aab")["a"] == 2
tmp = [5, 5, 6]
tmp.remove(5)
assert tmp == [5, 6]
assert Solution().rearrangeArray([2, 2, 2]) == [2, 2, 2]
assert Solution().rearrangeArray([3, 1, 2]) == [1, 2, 3]

# Bài 2
assert Solution().rearrangeArray([5, 2, 5, 2, 2, 9]) == [2, 5, 9, 2, 5, 2]
assert Solution().rearrangeArray([4, 4, 4, 1]) == [1, 4, 4, 4]

# Bài 3
def rearrange_todo_ans(nums):
    cnt = Counter(nums)
    keys = sorted(cnt)
    ans = []
    while keys:
        for x in keys:
            ans.append(x)
            cnt[x] -= 1
        keys = [x for x in keys if cnt[x] > 0]
    return ans

for nums, expected in examples:
    assert rearrange_todo_ans(nums) == expected

# Bài 4
def rearrange_pairs2(nums):
    cnt = Counter(nums)
    pairs = [(k, x) for x, c in cnt.items() for k in range(1, c + 1)]
    pairs.sort()
    return [x for _, x in pairs]

for nums, expected in examples:
    assert rearrange_pairs2(nums) == expected

# Bài 5(a)
def rearrange_desc2(nums):
    cnt = Counter(nums)
    keys = sorted(cnt, reverse=True)
    ans = []
    while keys:
        for x in keys:
            ans.append(x)
            cnt[x] -= 1
        keys = [x for x in keys if cnt[x] > 0]
    return ans

assert rearrange_desc2([3, 1, 3, 2, 1, 3]) == [3, 2, 1, 3, 1, 3]

# Bài 5(b)
def rounds_of(nums):
    cnt = Counter(nums)
    return [[x for x in sorted(cnt) if cnt[x] > k] for k in range(max(cnt.values()))]

def count_rounds(nums):
    return max(Counter(nums).values())

assert rounds_of([3, 1, 3, 2, 1, 3]) == [[1, 2, 3], [1, 3], [3]]
assert rounds_of([7, 7, 4, 4, 4]) == [[4, 7], [4, 7], [4]]
assert count_rounds([7, 7, 4, 4, 4]) == 3
print("PASS đáp án phần 19")

PASS đáp án phần 19


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (đếm tần suất, lấy một bản sao của mỗi giá trị mỗi vòng)

```python
from collections import Counter
from typing import List


class Solution:
    def rearrangeArray(self, nums: List[int]) -> List[int]:
        cnt = Counter(nums)
        keys = sorted(cnt)
        ans = []
        while keys:
            for x in keys:
                ans.append(x)
                cnt[x] -= 1
            keys = [x for x in keys if cnt[x] > 0]
        return ans
```

**Checklist**

- [ ] Mỗi vòng lấy **đúng một** bản sao của **mỗi giá trị khác nhau còn lại** (không phải mọi bản sao)?
- [ ] Trong mỗi vòng các giá trị **tăng dần** (`sorted(cnt)`, không dùng thứ tự khóa của `Counter`)?
- [ ] Không sửa list khi đang duyệt nó (không `remove` trong `for x in nums`)?
- [ ] Giá trị hết bản sao được loại khỏi vòng sau?
- [ ] Không sửa mảng đầu vào ngoài ý muốn?
- [ ] Đã thử: `n = 1`, mọi giá trị khác nhau, toàn một giá trị, tần suất khác nhau?
- [ ] Đã đối chiếu với mô phỏng trực tiếp trên input ngẫu nhiên?
- [ ] Skeleton đúng `class Solution` / `def rearrangeArray(self, nums: List[int]) -> List[int]`, có `import` cần thiết?